# Challenge Conversion Rate — Prédiction de l'abonnement à la newsletter

**Objectif :** prédire si un visiteur s'abonne à la newsletter **Data Science Weekly**, en restant strictement aligné avec le brief du projet et la méthodologie du cours.

## Ce que ce notebook cherche à démontrer
- une bonne compréhension du problème métier,
- une EDA utile et orientée décision,
- un pipeline scikit-learn propre,
- un choix de modèle justifié par la bonne métrique (**F1-score**),
- des recommandations métier lisibles et défendables.

## Logique méthodologique retenue
1. **EDA** pour comprendre la cible et les signaux métier.
2. **Prétraitement sklearn** avec `ColumnTransformer` et `Pipeline`.
3. **Modèle de base** avec `LogisticRegression`.
4. **Amélioration** via l'ajustement du seuil de décision, cohérent avec le F1-score demandé.
5. **Comparaison non linéaire** avec un modèle d'ensemble simple.
6. **Interprétation métier** du modèle final.

## Alignement avec le cours
Ce notebook réutilise directement la logique des notebooks suivants :
- `02-What_is_preprocessing.ipynb`
- `03-Preprocessing_with_sklearn.ipynb`
- `01-Logistic_Regression.ipynb`
- `01-Model_selection_evaluation.ipynb`
- `02-Cross_validation_and_grid_search.ipynb`
- `01-Regularization.ipynb`
- `01-Decision_trees.ipynb`
- `04-Ensemble_methods_template.ipynb`

> Idée clé : le but n'est pas de choisir l'algorithme le plus compliqué, mais la solution la plus **cohérente, robuste et interprétable**.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
    classification_report, roc_auc_score, average_precision_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'conversion_data_train.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
OUT_DIR = PROJECT_ROOT / 'artifacts'
OUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / 'conversion_data_train.csv'
TEST_PATH = DATA_DIR / 'conversion_data_test.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print('Train shape:', train.shape)
print('Test shape :', test.shape)
train.head()

## 1. Audit rapide des données


In [ ]:
audit = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'n_missing': train.isna().sum(),
    'n_unique': train.nunique()
})

audit

In [ ]:
print('Duplicate rows in train:', train.duplicated().sum())
print('Conversion rate:', round(train['converted'].mean() * 100, 2), '%')
print('\nTarget distribution:')
print(train['converted'].value_counts(normalize=True).rename('share'))

### Commentaire
Le dataset est **propre** : pas de valeurs manquantes, peu de colonnes, et une structure simple.

Deux points importants ressortent immédiatement :

- la cible est **fortement déséquilibrée** : seulement ~3,2 % de conversions ;
- le nombre de doublons exacts est élevé, mais cela ne veut pas forcément dire que les données sont sales.

Ici, le nombre de variables est faible et plusieurs colonnes ont une faible cardinalité. Il est donc normal que beaucoup de visiteurs partagent exactement le même profil observé (`country`, `age`, `new_user`, `source`, `total_pages_visited`, `converted`).

Conséquence méthodologique :
- l'**accuracy** serait trompeuse,
- la métrique à piloter doit être le **F1-score**, comme indiqué dans le brief,
- il n'y a pas de raison forte de supprimer les doublons, car ils reflètent aussi la distribution réelle des profils.


## 2. EDA orientée métier


In [ ]:
country_rates = train.groupby('country')['converted'].agg(['mean', 'count']).sort_values('mean', ascending=False)
source_rates = train.groupby('source')['converted'].agg(['mean', 'count']).sort_values('mean', ascending=False)
new_user_rates = train.groupby('new_user')['converted'].agg(['mean', 'count']).sort_values('mean', ascending=False)

print('Conversion by country')
display(country_rates)
print('Conversion by source')
display(source_rates)
print('Conversion by new_user')
display(new_user_rates)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

country_rates['mean'].plot(kind='bar', ax=axes[0], title='Conversion rate by country')
source_rates['mean'].plot(kind='bar', ax=axes[1], title='Conversion rate by source')
new_user_rates['mean'].plot(kind='bar', ax=axes[2], title='Conversion rate by new_user')

for ax in axes:
    ax.set_ylabel('conversion rate')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
summary_num = train.groupby('converted')[['age', 'total_pages_visited']].describe().T
summary_num

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train.boxplot(column='age', by='converted', ax=axes[0])
axes[0].set_title('Age by conversion')
axes[0].set_xlabel('converted')
axes[0].set_ylabel('age')

train.boxplot(column='total_pages_visited', by='converted', ax=axes[1])
axes[1].set_title('Pages visited by conversion')
axes[1].set_xlabel('converted')
axes[1].set_ylabel('total_pages_visited')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
pages_profile = train.groupby('total_pages_visited')['converted'].agg(['mean', 'count'])
pages_profile.head(12)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pages_profile.index, pages_profile['mean'])
plt.title('Conversion rate by number of pages visited')
plt.xlabel('total_pages_visited')
plt.ylabel('conversion rate')
plt.grid(True, alpha=0.3)
plt.show()

### Principaux enseignements de l'EDA
Les signaux métier sont très nets :

1. **`total_pages_visited` est le facteur dominant** : plus un utilisateur visite de pages, plus sa probabilité de conversion grimpe fortement.
2. **Les utilisateurs déjà revenus** (`new_user = 0`) convertissent bien mieux que les nouveaux visiteurs.
3. **L'âge agit négativement** : plus l'utilisateur est âgé, plus la conversion tend à baisser.
4. **Le pays compte** : `Germany` et `UK` performent mieux, `China` très nettement moins.
5. **La source a un effet secondaire**, mais bien moins fort que l'engagement mesuré par les pages vues.

Hypothèse métier : le vrai levier n'est pas uniquement l'acquisition. C'est surtout la **qualité du parcours visiteur** et la capacité à faire progresser l'utilisateur dans le site.


## 3. Prétraitement et séparation validation


In [ ]:
X = train.drop(columns='converted')
y = train['converted']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = ['country', 'source']
numeric_features = ['age', 'new_user', 'total_pages_visited']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

print(X_train.shape, X_valid.shape)

Ce choix est parfaitement aligné avec le cours : `Pipeline` + `ColumnTransformer` évitent les fuites de données et rendent le pipeline réutilisable sur le test final.


## 4. Modèle de base — Régression logistique


In [ ]:
baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=0.1, solver='liblinear', max_iter=1200))
])

baseline_model.fit(X_train, y_train)
valid_proba_lr = baseline_model.predict_proba(X_valid)[:, 1]
valid_pred_lr_default = (valid_proba_lr >= 0.5).astype(int)

baseline_default_metrics = pd.DataFrame({
    'metric': ['f1', 'precision', 'recall', 'roc_auc', 'avg_precision'],
    'value': [
        f1_score(y_valid, valid_pred_lr_default),
        precision_score(y_valid, valid_pred_lr_default, zero_division=0),
        recall_score(y_valid, valid_pred_lr_default, zero_division=0),
        roc_auc_score(y_valid, valid_proba_lr),
        average_precision_score(y_valid, valid_proba_lr)
    ]
})

baseline_default_metrics

### Lecture du modèle de base
La régression logistique classe bien les individus **en score** (`roc_auc`, `average_precision`), mais le seuil standard **0.5** est trop conservateur pour un problème aussi déséquilibré.

C'est exactement le type de situation où il faut **optimiser le seuil de décision** sur la métrique métier du challenge : le **F1-score**.


## 5. Amélioration du modèle — ajustement du seuil sur l'ensemble de validation


In [ ]:
def find_best_threshold(y_true, proba, start=0.01, stop=0.99, step=0.005):
    thresholds = np.arange(start, stop + step, step)
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        rows.append({
            'threshold': round(float(t), 4),
            'f1': f1_score(y_true, pred, zero_division=0),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0)
        })
    scores = pd.DataFrame(rows)
    best_row = scores.loc[scores['f1'].idxmax()]
    return scores, best_row

threshold_scores_lr, best_lr = find_best_threshold(y_valid, valid_proba_lr)
best_lr

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(threshold_scores_lr['threshold'], threshold_scores_lr['f1'], label='F1-score')
plt.plot(threshold_scores_lr['threshold'], threshold_scores_lr['precision'], label='Precision', alpha=0.8)
plt.plot(threshold_scores_lr['threshold'], threshold_scores_lr['recall'], label='Recall', alpha=0.8)
plt.axvline(best_lr['threshold'], linestyle='--')
plt.title('Threshold tuning — Logistic Regression')
plt.xlabel('decision threshold')
plt.ylabel('score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
best_threshold_lr = float(best_lr['threshold'])
valid_pred_lr_tuned = (valid_proba_lr >= best_threshold_lr).astype(int)

lr_results = {
    'model': 'Logistic Regression',
    'f1_default_0.5': f1_score(y_valid, valid_pred_lr_default),
    'best_threshold': best_threshold_lr,
    'f1_tuned': f1_score(y_valid, valid_pred_lr_tuned),
    'precision_tuned': precision_score(y_valid, valid_pred_lr_tuned, zero_division=0),
    'recall_tuned': recall_score(y_valid, valid_pred_lr_tuned, zero_division=0)
}

pd.DataFrame([lr_results])

In [ ]:
cm_lr = confusion_matrix(y_valid, valid_pred_lr_tuned)
pd.DataFrame(cm_lr, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1'])

### Résultat clé
Le gain principal du projet vient ici : **on garde un modèle simple et interprétable, mais on l'utilise intelligemment**.

L'ajustement du seuil permet de passer d'un F1 médiocre au seuil 0,5 à un **F1 très compétitif** en validation.
Cette amélioration est méthodologiquement propre et cohérente avec la métrique officielle du challenge.


### Robustesse du choix retenu
Une vérification complémentaire en validation croisée légère conduit au même constat :

- la famille **régression logistique** reste très stable,
- les valeurs de `C` testées donnent des performances proches,
- la zone optimale pour le seuil se situe globalement autour de **0,38 - 0,44**.

Autrement dit, la performance du modèle ne repose pas sur un réglage fragile, mais sur un **signal réellement robuste** dans les données.


## 6. Comparaison non linéaire — AdaBoost


In [ ]:
adaboost_model = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])),
    ('model', AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42))
])

adaboost_model.fit(X_train, y_train)
valid_proba_ada = adaboost_model.predict_proba(X_valid)[:, 1]
valid_pred_ada_default = (valid_proba_ada >= 0.5).astype(int)
threshold_scores_ada, best_ada = find_best_threshold(y_valid, valid_proba_ada)
valid_pred_ada_tuned = (valid_proba_ada >= float(best_ada['threshold'])).astype(int)

ada_results = {
    'model': 'AdaBoost',
    'f1_default_0.5': f1_score(y_valid, valid_pred_ada_default),
    'best_threshold': float(best_ada['threshold']),
    'f1_tuned': f1_score(y_valid, valid_pred_ada_tuned),
    'precision_tuned': precision_score(y_valid, valid_pred_ada_tuned, zero_division=0),
    'recall_tuned': recall_score(y_valid, valid_pred_ada_tuned, zero_division=0)
}

comparison = pd.DataFrame([lr_results, ada_results]).sort_values('f1_tuned', ascending=False)
comparison

### Choix du modèle
Le benchmark confirme que la meilleure version du projet est la suivante :

- **Modèle retenu** : `LogisticRegression(C=0.1)`
- **Seuil retenu** : `0.44`
- **Pourquoi ?**
  - meilleur F1-score sur validation après ajustement du seuil,
  - performance supérieure au comparaison non linéaire testé,
  - coefficients directement interprétables,
  - parfaite cohérence avec la demande du brief : **prédire** et **expliquer**.

Conclusion : **plus simple sur l'algorithme, plus fort sur la méthode**.


## 7. Entraînement final sur tout le train et prédictions sur le jeu de test


In [ ]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=0.1, solver='liblinear', max_iter=1200))
])

final_model.fit(X, y)

test_proba = final_model.predict_proba(test)[:, 1]
test_pred = (test_proba >= best_threshold_lr).astype(int)

submission = pd.DataFrame({'converted': test_pred})
submission_path = OUT_DIR / 'conversion_rate_test_predictions_premium.csv'
submission.to_csv(submission_path, index=False)

print('Saved submission to:', submission_path)
submission.head()

## 8. Interpréter le modèle final et en extraire la valeur métier


In [ ]:
feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = final_model.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

coef_path = OUT_DIR / 'logistic_coefficients_premium.csv'
coef_df.to_csv(coef_path, index=False)
coef_df.head(10)

In [ ]:
top_positive = coef_df.sort_values('coefficient', ascending=False).head(8)
top_negative = coef_df.sort_values('coefficient', ascending=True).head(8)

print('Top positive effects')
display(top_positive)
print('Top negative effects')
display(top_negative)

### Interprétation métier
Les enseignements les plus importants sont :

- **Visiter plus de pages** augmente très fortement la probabilité de conversion.
- Être **déjà connu** du site (`new_user = 0`) aide beaucoup.
- Les visiteurs plus âgés convertissent moins.
- `China` est un segment nettement plus difficile à convertir.
- `Germany` et `UK` sont plus favorables.

Attention : les coefficients ne prouvent pas une causalité pure. Mais ils donnent des **leviers d'action plausibles** et utilisables.


In [ ]:
recommendations = pd.DataFrame({
    'priority': [1, 2, 3, 4],
    'recommendation': [
        "Augmenter le nombre de pages utiles visitées avant l'écran d'inscription",
        "Créer des parcours spécifiques pour les nouveaux visiteurs",
        "Personnaliser davantage les messages selon le pays",
        "Tester des CTA et contenus adaptés aux profils plus âgés"
    ],
    'why': [
        "total_pages_visited est de très loin le signal le plus fort",
        "les returning users convertissent beaucoup mieux que les nouveaux",
        "les taux de conversion diffèrent fortement selon le pays",
        "l'âge semble corrélé négativement à la conversion"
    ]
})
recommendations

## 9. Conclusion finale


### Conclusion du projet
Ce notebook propose une réponse complète et cohérente au challenge :

- brief respecté,
- EDA claire,
- prétraitement sklearn propre,
- modèle de base cohérent,
- amélioration réelle du score via l'ajustement du seuil,
- comparaison non linéaire,
- choix final justifié,
- fichier de soumission généré,
- recommandations métier explicites.

### Message principal
Le projet montre qu'un modèle simple, correctement prétraité, évalué avec la bonne métrique et utilisé avec un seuil adapté, peut fournir une solution à la fois performante, robuste et interprétable.

C'est ce qui fait ici la qualité de la démarche : une réponse sobre, lisible et techniquement défendable.


In [ ]:
summary = {
    'train_shape': tuple(train.shape),
    'test_shape': tuple(test.shape),
    'conversion_rate': float(train['converted'].mean()),
    'logreg_f1_default_0.5': float(lr_results['f1_default_0.5']),
    'logreg_best_threshold': float(lr_results['best_threshold']),
    'logreg_f1_tuned': float(lr_results['f1_tuned']),
    'logreg_precision_tuned': float(lr_results['precision_tuned']),
    'logreg_recall_tuned': float(lr_results['recall_tuned']),
    'adaboost_f1_tuned': float(ada_results['f1_tuned']),
    'submission_path': 'artifacts/conversion_rate_test_predictions.csv',
    'coefficients_path': 'artifacts/logistic_coefficients.csv'
}

summary_path = OUT_DIR / 'project_summary.json'
pd.Series(summary).to_json(summary_path, indent=2)
print(summary)